In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

In [ ]:
class DatabaseManager:
    def __init__(self, db_user, db_password, db_host, db_port, db_name):
        self.connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
        self.engine = create_engine(self.connection_string)
        
    def get_engine(self):
        return self.engine

In [ ]:
class DataQualityManager:
    def __init__(self, df):
        self.df = df.copy()
        self.audit_log = []
    
    def log_action(self, step, level, taxonomy, column, action, reason):
        """Registra ogni operazione nell'Audit Log come richiesto dalla compliance."""
        self.audit_log.append({
            'Timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
            'Step': step,
            'Taxonomy_Level': level,
            'Taxonomy_Type': taxonomy,
            'Column': column,
            'Action': action,
            'Reason': reason
        })

    def run_profiling(self):
        """1. Data Profiling: Calcola gli indici di Completeness e Uniqueness."""
        print("--- DATA PROFILING (Pre-Load) ---")
        
        # Uniqueness (Controllo su eventid)
        total_rows = len(self.df)
        unique_events = self.df['eventid'].nunique()
        uniqueness_score = unique_events / total_rows
        print(f"Uniqueness (eventid): {uniqueness_score:.2%} ({unique_events}/{total_rows})")
        
        # Validity (Controllo se ci sono -9 o -99 mascherati nei numerici)
        metric_cols = ['nkill', 'nwound', 'propvalue']
        for col in metric_cols:
            if col in self.df.columns:
                invalid_count = self.df[self.df[col].isin([-9, -99])].shape[0]
                if invalid_count > 0:
                    print(f"Validity Alert [{col}]: Trovati {invalid_count} valori illegali (-9/-99).")

        # Completeness
        print("\nCompleteness per colonne chiave:")
        for col in ['country_txt', 'region_txt', 'gname', 'attacktype1_txt']:
            if col in self.df.columns:
                completeness = self.df[col].notnull().sum() / total_rows
                print(f"- {col}: {completeness:.2%}")
        print("-" * 33 + "\n")

    def clean_noisy_data(self):
        """2 & 3. Trattamento Noisy Data: Sostituzione dei valori di default errati."""
        cols_to_clean = ['nkill', 'nkillter', 'nwound', 'propvalue']
        for col in cols_to_clean:
            if col in self.df.columns:
                mask = self.df[col].isin([-9, -99])
                cleaned_count = mask.sum()
                if cleaned_count > 0:
                    self.df.loc[mask, col] = np.nan
                    self.log_action(
                        step="clean_noisy_data",
                        level="Instance Level",
                        taxonomy="Noisy Data / Missing mascherati",
                        column=col,
                        action="Sostituzione -9/-99 con NaN",
                        reason=f"Standardizzazione missing values. Righe modificate: {cleaned_count}"
                    )

    def enforce_consistency(self):
        """4. Coerenza Semantica: Applicazione Business Rules."""
        if 'nkill' in self.df.columns and 'nkillter' in self.df.columns:
            inconsistent_mask = self.df['nkillter'] > self.df['nkill']
            inconsistent_count = inconsistent_mask.sum()
            
            if inconsistent_count > 0:
                self.df.loc[inconsistent_mask, 'nkillter'] = self.df.loc[inconsistent_mask, 'nkill']
                self.log_action(
                    step="enforce_consistency",
                    level="Instance Level",
                    taxonomy="Inconsistent Data",
                    column="nkillter",
                    action="nkillter allineato a nkill",
                    reason=f"Violazione logica: perpetratori uccisi > morti totali. Fissate {inconsistent_count} righe."
                )

    def apply_smoothing_binning(self):
        """3. Smoothing (Dimostrazione Teorica su propvalue)."""
        if 'propvalue' in self.df.columns:
            valid_mask = self.df['propvalue'] > 0
            
            if valid_mask.sum() > 10: # Solo se ci sono abbastanza dati
                # Creazione di 4 bin equifrequenti usando qcut
                try:
                    self.df['propvalue_binned'] = pd.qcut(self.df.loc[valid_mask, 'propvalue'], q=4, labels=False, duplicates='drop')
                    
                    # Sostituzione dei valori estremi con la media del proprio bin (Smoothing)
                    bin_means = self.df.groupby('propvalue_binned')['propvalue'].transform('mean')
                    self.df.loc[valid_mask, 'propvalue_smoothed'] = bin_means
                    
                    self.log_action(
                        step="apply_smoothing",
                        level="Instance Level",
                        taxonomy="Noisy Outliers",
                        column="propvalue",
                        action="Equi-depth Binning & Mean Smoothing",
                        reason="Riduzione della varianza sui valori delle proprietà colpite."
                    )
                except ValueError:
                    pass
    
    def apply_midpoint_date_logic(self):
        """Implementazione della Midpoint Logic per le date approssimative (Option B)."""
        self.df['is_approximate_date'] = False
        
        # Caso 1: Mese mancante (0) -> Metà Anno (1 Luglio)
        missing_month_mask = self.df['imonth'] == 0
        if missing_month_mask.sum() > 0:
            self.df.loc[missing_month_mask, 'is_approximate_date'] = True
            self.df.loc[missing_month_mask, 'imonth'] = 7
            self.df.loc[missing_month_mask, 'iday'] = 1
            
            self.log_action(
                step="apply_midpoint_date", level="Instance Level", taxonomy="Missing Data",
                column="imonth, iday", action="Imputazione Midpoint (1 Luglio)",
                reason=f"Mese mancante (valore 0). Sostituiti {missing_month_mask.sum()} record."
            )

        # Caso 2: Solo Giorno mancante (0) -> Metà Mese (15)
        missing_day_mask = (self.df['iday'] == 0) & (self.df['imonth'] != 0)
        if missing_day_mask.sum() > 0:
            self.df.loc[missing_day_mask, 'is_approximate_date'] = True
            self.df.loc[missing_day_mask, 'iday'] = 15
            
            self.log_action(
                step="apply_midpoint_date", level="Instance Level", taxonomy="Missing Data",
                column="iday", action="Imputazione Midpoint (Giorno 15)",
                reason=f"Giorno mancante (valore 0). Sostituiti {missing_day_mask.sum()} record."
            )

        # Generazione della colonna full_date per Tableau
        date_strings = (
            self.df['iyear'].astype(str) + '-' + 
            self.df['imonth'].astype(str).str.zfill(2) + '-' + 
            self.df['iday'].astype(str).str.zfill(2)
        )
        self.df['full_date'] = pd.to_datetime(date_strings, errors='coerce')
    
    def handle_missing_metrics(self):
        """Gestione statistica dei Missing Data per le metriche principali."""
        
        # 1. nkillter (Terroristi Uccisi) -> Strategia MNAR
        if 'nkillter' in self.df.columns:
            self.df['nkillter_reported'] = self.df['nkillter'].notnull().astype(int)
            
            missing_nkillter = self.df['nkillter'].isnull().sum()
            self.log_action(
                step="handle_missing_metrics", level="Schema Level", taxonomy="MNAR",
                column="nkillter_reported", action="Creazione Flag Indicator",
                reason=f"Dato MNAR. Evitata imputazione per {missing_nkillter} record, creato flag."
            )

        # 2. nkill e nwound -> Strategia MAR
        # Uso la mediana raggruppata per Regione e Tipo di Attacco
        for col in ['nkill', 'nwound']:
            if col in self.df.columns:
                missing_mask = self.df[col].isnull()
                missing_count = missing_mask.sum()
                
                if missing_count > 0:
                    # Calcolo mediana condizionata (Regione + Arma/Attacco)
                    grouped_medians = self.df.groupby(['region', 'attacktype1'])[col].transform('median')
                    self.df.loc[missing_mask, col] = grouped_medians[missing_mask]
                    
                    # Fallback 1: Se un intero cluster è nullo, si scala alla mediana della sola Regione
                    still_missing = self.df[col].isnull()
                    if still_missing.sum() > 0:
                        region_medians = self.df.groupby('region')[col].transform('median')
                        self.df.loc[still_missing, col] = region_medians[still_missing]
                    
                    # Fallback 2: Sicurezza globale se restano rarissimi null (mediana globale)
                    final_missing = self.df[col].isnull()
                    if final_missing.sum() > 0:
                        self.df.loc[final_missing, col] = self.df[col].median()
                        
                    self.log_action(
                        step="handle_missing_metrics", level="Instance Level", taxonomy="MAR",
                        column=col, action="Imputazione Mediana Condizionata",
                        reason=f"Riempiti {missing_count} nulli usando cluster (Region + AttackType)."
                    )

    def plot_outliers(self):
        """3. Rilevamento Outliers: Valid vs Noisy tramite Scatter Plot Dinamico."""
        if 'nkill' in self.df.columns and 'nwound' in self.df.columns:
            plt.figure(figsize=(10, 6))
            
            sns.scatterplot(data=self.df, x='nkill', y='nwound', alpha=0.5, color='blue')
            plt.title('Scatter Plot Diagnostico: Morti vs Feriti (Valid vs Noisy Outliers)')
            plt.xlabel('Numero di Vittime (nkill)')
            plt.ylabel('Numero di Feriti (nwound)')
            
            max_kill_value = self.df['nkill'].max()
            
            percentile_999 = self.df['nkill'].quantile(0.999)
            
            plt.axvline(x=percentile_999, color='orange', linestyle='--', 
                        label=f'Soglia 99.9° Percentile ({int(percentile_999)} vittime)')
            
            plt.axvline(x=max_kill_value, color='red', linestyle='-', 
                        label=f'Max Valid Outlier Reale ({int(max_kill_value)} vittime)')
            
            plt.legend()
            plt.show()

    def get_processed_data_and_log(self):
        """Esegue l'intera pipeline DQA rispettando l'ordine delle dipendenze."""
        self.run_profiling()                  # 1. Analisi metadati
        self.clean_noisy_data()               # 2. Converte -9/-99 in NaN
        self.apply_midpoint_date_logic()      # 3. Risolve le date approssimative
        self.handle_missing_metrics()         # 4. Imputa MAR (nkill/nwound) e flagga MNAR (nkillter)
        self.enforce_consistency()            # 5. Verifica contraddizioni di business rule
        self.apply_smoothing_binning()        # 6. Riduce la varianza (Smoothing)
        self.plot_outliers()                  # 7. Diagnostica grafica
        
        audit_df = pd.DataFrame(self.audit_log)
        return self.df, audit_df

In [ ]:
class DataProcessor:
    def __init__(self, file_path):
        self.file_path = file_path
        self.columns_to_load = [
            'eventid', 'iyear', 'imonth', 'iday', 'success', 'suicide', 'nkill', 'nkillter', 'nwound', 'propvalue',
            'country', 'country_txt', 'region', 'region_txt', 'provstate', 'city', 'latitude', 'longitude',
            'gname', 'gsubname', 'claimed', 'attacktype1', 'attacktype1_txt', 
            'weaptype1', 'weaptype1_txt', 'targtype1', 'targtype1_txt'
        ]

    def load_data_staging(self, engine):
        print("1. Lettura dati raw...")
        df = pd.read_excel(self.file_path, usecols=self.columns_to_load)

        print("2. Inizio Data Quality Assurance Pipeline...")
        dq_manager = DataQualityManager(df)
        df_cleaned, audit_log_df = dq_manager.get_processed_data_and_log()
        
        print("\n3. Salvataggio Audit Log su file CSV (Compliance)...")
        audit_log_df.to_csv('./dataset/audit_log.csv', index=False)

        print("4. Caricamento in Staging (PostgreSQL)...")
        df_cleaned.to_sql('raw_gtd', con=engine, schema='staging', if_exists='replace', index=False)

In [ ]:
class DataPipelineManager:
    def __init__(self, engine):
        self.engine = engine

    def execute_sql_file(self, file_path):
        """Metodo utility per leggere ed eseguire script SQL da file."""
        with open(file_path, 'r', encoding='utf-8') as file:
            sql_script = file.read()
        
        queries = [q.strip() for q in sql_script.split(';') if q.strip()]
        with self.engine.begin() as conn:
            for query in queries:
                conn.execute(text(query))

    def run_full_pipeline(self, rd_path, dwh_path):
        """Orchestra la fase ELT interamente in-database."""
        print("4. Popolamento Reconciled Database (SQL)...")
        self.execute_sql_file(rd_path)
        
        print("5. Esecuzione ETL verso Data Warehouse (SQL)...")
        self.execute_sql_file(dwh_path)

    def export_star_schema_for_tableau(self, export_path):
        """Esporta la tabella denormalizzata applicando controlli qualitativi finali."""
        print("6. Esportazione One Big Table per Tableau...")
        with open(export_path, 'r', encoding='utf-8') as file:
            query = file.read().strip()

        with self.engine.connect() as conn:
            result = conn.execute(text(query))
            df_one_big_table = pd.DataFrame(result.fetchall(), columns=result.keys())
            
            metric_cols = ['nkill', 'nkillter', 'nwound', 'nkillter_reported']
            for col in metric_cols:
                if col in df_one_big_table.columns:
                    df_one_big_table[col] = pd.to_numeric(df_one_big_table[col], errors='coerce').astype('Int64')

            df_one_big_table.to_csv('./dataset/one_big_table.csv', index=False)
        print("Pipeline completata con successo! Dati pronti per Tableau.")

In [ ]:
def main(already_populated=False):
    DB_USER = 'aldogioia'
    DB_PASS = 'rafele'
    DB_HOST = 'localhost'
    DB_PORT = '5432'
    DB_NAME = 'data_warehouse_project'
    
    FILE_PATH = 'dataset/globalterrorismdb_0522dist.xlsx'
    
    ELT_RD_PATH = 'sql/populate_rd.sql'
    ETL_DWH_PATH = 'sql/etl_queries.sql'
    EXPORT_PATH = 'sql/one_big_table.sql'

    db_manager = DatabaseManager(DB_USER, DB_PASS, DB_HOST, DB_PORT, DB_NAME)
    engine = db_manager.get_engine()
    pipeline_manager = DataPipelineManager(engine)

    if not already_populated:
        # Phase 1: E-T-L con Python
        processor = DataProcessor(FILE_PATH)
        
        # Creiamo lo schema di staging se non esiste
        with engine.begin() as conn:
            conn.execute(text("CREATE SCHEMA IF NOT EXISTS staging;"))
            
        processor.load_data_staging(engine)

        # Phase 2: E-L-T con SQL
        pipeline_manager.run_full_pipeline(ELT_RD_PATH, ETL_DWH_PATH)

    # Phase 3: Esportazione
    pipeline_manager.export_star_schema_for_tableau(EXPORT_PATH)

if __name__ == "__main__":
    main(False)